In [ ]:
!git clone -q https://github.com/cad0m/jarida-rag.git

In [ ]:
# Run this after you push new code (after the imports cell has run once)
!git -C jarida-rag pull

import importlib
from jarida_rag import chunk, config, evaluate, pdf, rag, store

for module in [config, pdf, chunk, store, rag, evaluate]:
    importlib.reload(module)

In [ ]:
!pip install -q "transformers>=4.51" sentence-transformers qdrant-client mlflow pypdfium2 langchain-text-splitters bitsandbytes

In [ ]:
import os
import sys

from kaggle_secrets import UserSecretsClient

DAGSHUB_USER = "cad0m"

os.environ["JARIDA_PDF_DIR"] = "/kaggle/input/datasets/ayouberraouy/jarida"
os.environ["JARIDA_WORK"] = "/kaggle/working"
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{DAGSHUB_USER}/jarida-rag.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USER
os.environ["MLFLOW_TRACKING_PASSWORD"] = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
sys.path.insert(0, "/kaggle/working/jarida-rag/src")

import mlflow
import torch
from jarida_rag import config, evaluate, rag, store

mlflow.set_tracking_uri(config.MLFLOW_URI)
mlflow.set_experiment("gazette-rag")


def show_gpus():
    for gpu in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(gpu)
        print(f"GPU {gpu}: {free / 1024**3:.1f} GB free of {total / 1024**3:.1f} GB")


show_gpus()

In [ ]:
client = store.open_store()
embedder = store.load_embedder()
print(store.build_index(client, embedder), "chunks")
show_gpus()

In [ ]:
tokenizer, model = rag.load_llm()
print(f"LLM size: {model.get_memory_footprint() / 1024**3:.1f} GB")
show_gpus()

In [ ]:
questions = [
    "Which decree regulates marine aquaculture farms?",
    "What is the legal age of marriage in Morocco?",
]

for question in questions:
    reply, hits = rag.answer(question, client, embedder, tokenizer, model)
    print(question)
    print(reply)
    for hit in hits:
        print(f"  - BO {hit['issue']} p.{hit['page_start']} | {hit['title'][:70]}")
    print()

In [ ]:
evaluate.evaluate(client, embedder, run_name="baseline")